In [1]:
import torch

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = { s:i+1 for i,s in enumerate(chars)} # mapping character to numbers, e.g. A = 1, B = 2, etc
stoi['.'] = 0 # manually Map . to 0
itos = { i:s for s,i in stoi.items()} # reverse mapping integers to chars

In [22]:
# Create the training set of Trigrams (x1, x2, y)

x1s, x2s, ys = [], [], []

for w in words:
    chs = ['.'] + ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        print(ch1, ch2, ch3)
        x1s.append(ix1)
        x2s.append(ix2)
        ys.append(ix3)



x1s = torch.tensor(x1s)
x2s = torch.tensor(x2s)
ys = torch.tensor(ys)
num = x1s.nelement()
print('Number of Examples:', num)

. . e
. e m
e m m
m m a
m a .
. . o
. o l
o l i
l i v
i v i
v i a
i a .
. . a
. a v
a v a
v a .
. . i
. i s
i s a
s a b
a b e
b e l
e l l
l l a
l a .
. . s
. s o
s o p
o p h
p h i
h i a
i a .
. . c
. c h
c h a
h a r
a r l
r l o
l o t
o t t
t t e
t e .
. . m
. m i
m i a
i a .
. . a
. a m
a m e
m e l
e l i
l i a
i a .
. . h
. h a
h a r
a r p
r p e
p e r
e r .
. . e
. e v
e v e
v e l
e l y
l y n
y n .
. . a
. a b
a b i
b i g
i g a
g a i
a i l
i l .
. . e
. e m
e m i
m i l
i l y
l y .
. . e
. e l
e l i
l i z
i z a
z a b
a b e
b e t
e t h
t h .
. . m
. m i
m i l
i l a
l a .
. . e
. e l
e l l
l l a
l a .
. . a
. a v
a v e
v e r
e r y
r y .
. . s
. s o
s o f
o f i
f i a
i a .
. . c
. c a
c a m
a m i
m i l
i l a
l a .
. . a
. a r
a r i
r i a
i a .
. . s
. s c
s c a
c a r
a r l
r l e
l e t
e t t
t t .
. . v
. v i
v i c
i c t
c t o
t o r
o r i
r i a
i a .
. . m
. m a
m a d
a d i
d i s
i s o
s o n
o n .
. . l
. l u
l u n
u n a
n a .
. . g
. g r
g r a
r a c
a c e
c e .
. . c
. c h
c h l
h l o
l o 

In [23]:
x1s

tensor([ 0,  0,  5,  ..., 26, 25, 26])

In [24]:
x2s

tensor([ 0,  5, 13,  ..., 25, 26, 24])

In [25]:
ys

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [26]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad = True) # 54 beacuse we concatnate x1s and x2s

In [27]:
import torch.nn.functional as F

x1enc = F.one_hot(x1s, num_classes= 27).float() 
x2enc = F.one_hot(x2s, num_classes= 27).float() 
xenc = x1enc + x2enc # input to the netwrok : one-hot encoding concat x1s and x2s total 
x2enc.shape

torch.Size([228146, 27])

In [37]:
import torch.nn.functional as F

# Gradient Descent

for k in range(100):
    # Forward Pass
    x1enc = F.one_hot(x1s, num_classes= 27).float() 
    x2enc = F.one_hot(x2s, num_classes= 27).float() 
    xenc = torch.cat([x1enc, x2enc], dim=1) # input to the network : one-hot encoding concat x1s and x2s total , dim =1 indictes concatenation through cols
    logits = xenc @ W # predicts log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdim = True) #probablities for next character
    loss = -probs[torch.arange(num), ys].log().mean() # negative loss likelihood
    print(loss.item())

    # Backward Pass
    W.grad = None # set to zero the gradient
    loss.backward()

    #Update
    W.data += -50 * W.grad


2.3874449729919434
2.373255729675293
2.4233241081237793
2.3636093139648438
2.3873844146728516
2.3731913566589355
2.4232559204101562
2.3635478019714355
2.3873255252838135
2.3731281757354736
2.4231889247894287
2.363487720489502
2.387268543243408
2.3730664253234863
2.423124313354492
2.363428831100464
2.387211322784424
2.3730061054229736
2.423060894012451
2.3633711338043213
2.387155771255493
2.3729469776153564
2.4229984283447266
2.363314390182495
2.387101411819458
2.372889518737793
2.422938108444214
2.363259792327881
2.3870487213134766
2.372833013534546
2.4228782653808594
2.363205671310425
2.3869972229003906
2.3727777004241943
2.4228200912475586
2.3631529808044434
2.386946678161621
2.3727238178253174
2.422762632369995
2.3631017208099365
2.386897563934326
2.3726704120635986
2.422706365585327
2.363051414489746
2.386850357055664
2.3726186752319336
2.4226510524749756
2.363001823425293
2.3868026733398438
2.372567892074585
2.422597646713257
2.3629534244537354
2.386756181716919
2.3725180625915527

In [42]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):

    out = []
    ix1 = 0
    ix2 = 0
    while True:
        # ----------
        # Before:
        #p= P[ix]
        # ----------
        # Now:
        x1enc = F.one_hot(torch.tensor(ix1), num_classes=27).float()   # shape (27,)
        x2enc = F.one_hot(torch.tensor(ix2), num_classes=27).float()   # shape (27,)
        xenc = torch.cat([x1enc, x2enc], dim=0)          # shape (54,) — dim=0 now, not dim=1
        logits = xenc @ W                                 # shape (27,)
        counts = logits.exp()
        p = counts / counts.sum()                         # shape (27,)
        # ----------
        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        ix1, ix2 = ix2, ix3        
        out.append(itos[ix3])
        if ix3 == 0:
            break
    print(''.join(out))

cexze.
moullurailaziaydamellimittain.
lusan.
ka.
da.
